In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/01")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/02")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/03")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/04")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/05")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/06")
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/07")

In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader")

In [0]:
dbutils.fs.cp("/Volumes/samples/databricks/datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv","/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/01")

dbutils.fs.cp("/Volumes/samples/databricks/datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv","/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/02")

dbutils.fs.cp("/Volumes/samples/databricks/datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv","/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/03")

In [0]:
# File Detection Modes of Autoloader
# - Directory Listing (uses API calls to detect new files)
#   - Autoloader identifies files and ensures that their metadata is saved in a scalable key-value store (RocksDB)
#.    in its pipeline's checkpoint location
# - File Notification (uses Notification and Queue Services - requires elevated cloud permission for setup)
#   - Note : Notification service would put a file event with metadata in the Queue for Auto Loader to subscribe and consume. 

In [0]:
# How to use AutoLoader?
# - AutoLoader uses a structured streaming source 'cloudFiles' in order to process the data from cloud storage location.
# - Since cloudFiles is a structured streaming source we can go ahead and use spark.readStrema in order to read files from cloud storage location using autoloader.
# - We can pass schema as well but SchemaHints allows us to write only those columns for which we want change.
 
df = ( spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("pathGlobFilter","*.csv")
        .option("header","true")
        .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
        .option("cloudFiles.schemaLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/1/") 
        .load("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/*/")
)

In [0]:
from pyspark.sql.functions import col

In [0]:
# Write data to delta lake - dbcertification.easewithdata.invoice_al_1
# path : abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoice_al_1
(
    df
    .withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/1/")
    .outputMode("append")
    .trigger(availableNow = True)
    .option("path","abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoice_al_1")
    .table("dbcertification.easewithdata.invoice_al_1")
)

In [0]:
%sql
select * from dbcertification.easewithdata.invoice_al_1;

In [0]:
%sql
select __file, count(*)
from dbcertification.easewithdata.invoice_al_1
group by __file;  

In [0]:
# Re-Run write command : to see if code reads the files again and re-run the command again.
(
    df
    .withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/1/")
    .outputMode("append")
    .trigger(availableNow = True)
    .option("path","abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoice_al_1")
    .table("dbcertification.easewithdata.invoice_al_1")
)

In [0]:
%sql
select __file, count(*)
from dbcertification.easewithdata.invoice_al_1
group by __file;  

#### Add one more file and process it

In [0]:
dbutils.fs.cp("/Volumes/samples/databricks/datasets/definitive-guide/data/retail-data/by-day/2010-12-05.csv","/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/2010/12/05")

In [0]:
# Re-Run write command : to see if code reads the files again and re-run the command again.
(
    df
    .withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/1/")
    .outputMode("append")
    .trigger(availableNow = True)
    .option("path","abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoice_al_1")
    .table("dbcertification.easewithdata.invoice_al_1")
)

#### It only processed the data Incrementally

In [0]:
%sql
select __file, count(*)
from dbcertification.easewithdata.invoice_al_1
group by __file;  

##### How Schema Evolution Works with AutoLoaders
- AutoLoader gives you benefit to handle schema that comes with your files
- Schema Evolution  -> Add New Columns
    - Stream Fails. New Columns are added to the schema. Existing columns do not evolve data types.
- There are 4 different modes that you can use in order to handle the files or the schema changes or the evolution that happens when data arrives in cloud location.
- Default : StreamFails. addNewColumns
- rescue :
- failOnNewColumns : Stream Fails have to go and change schema manually.
- none : 

Default : StreamFails. addNewColumns

In [0]:
df = ( 
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("pathGlobFilter","*.csv")
    .option("header","true")
    .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
    .option("cloudFiles.schemaLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/1/") 
    .load("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/*/")
)

In [0]:
(
    df
    .withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/1/")
    .option("mergeSchema",True)
    .outputMode("append")
    .trigger(availableNow = True)
    .option("path","abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoice_al_1")
    .table("dbcertification.easewithdata.invoice_al_1")
)

In [0]:
%sql
select __file, count(*)
from dbcertification.easewithdata.invoice_al_1
group by __file;  

rescue :
- New Columns will be put to column rescued data.

In [0]:
df = ( 
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("pathGlobFilter","*.csv")
    .option("header","true")
    .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
    .option("cloudFiles.schemaLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/2/") 
    .option("cloudFiles.schemaEvolutionMode","rescue")
    .load("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/*/")
)

In [0]:
(
    df
    .withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/2/")
    .option("mergeSchema",True)
    .outputMode("append")
    .trigger(availableNow = True)
    .option("path","abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoice_al_2")
    .table("dbcertification.easewithdata.invoice_al_2")
)

In [0]:
%sql
select __file, count(*)
from dbcertification.easewithdata.invoice_al_2
group by __file;

In [0]:
%sql
select * from dbcertification.easewithdata.invoice_al_2 where `_rescued_data` is not null;

Schema Evolution : None
- Default mode if schema is provided.
- Does not evolve the schema, new columns are ignored and data is not rescued unless the rescuedDataColumn option is set.
- Stream does not fail due to schema changes.


In [0]:
df = ( 
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("pathGlobFilter","*.csv")
    .option("header","true")
    .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
    .option("cloudFiles.schemaLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/3/") 
    .option("cloudFiles.schemaEvolutionMode","none")
    .load("/Volumes/dbcertification/easewithdata/volume_easewithdata/autoloader_input/*/")
)

In [0]:
(
    df
    .withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/checkpoint/autoloader/3/")
    .option("mergeSchema",True)
    .outputMode("append")
    .trigger(availableNow = True)
    .option("path","abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoice_al_3")
    .table("dbcertification.easewithdata.invoice_al_3")
)

In [0]:
%sql
select __file, count(*)
from dbcertification.easewithdata.invoice_al_3 
group by __file; 

In [0]:
%sql
select * from dbcertification.easewithdata.invoice_al_3 -- where `_rescued_data` is not null;

FailOnNewColumn : Whenever new column is added stream fails to load data.